# Where embeddings live

**What you'll learn.** embpy always writes embeddings into a predictable,
AnnData-native place, records how they were made, and leaves your `.X` alone.
Once you know the contract, every embedding is easy to find, trust, and reload.

Three rules:

- **row-aligned** embeddings → `.obsm` (one vector per observation)
- **feature-aligned** embeddings → `.varm` (one vector per variable)
- **payload + provenance** → `.uns` (the entity-level vectors and how they were made)

`.X` is reserved for your counts / expression and is never overwritten.

In [1]:
import os

# The text encoder used later tokenises before a fork, which makes HuggingFace
# print a parallelism warning over the cell outputs. Nothing here is
# throughput-bound, so turn it off rather than read around it.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder

embedder = BioEmbedder(device="auto", organism="human")
genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1"]

## Row-aligned embeddings → `.obsm`

When each **row** of your AnnData is the thing you're embedding — a gene, a
cell, a sample — the embedding is row-aligned and lands in `.obsm`, exactly
like a PCA or UMAP basis. It has one vector per `obs`, matched to `obs_names`.

In [2]:
rows = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)
rows = embedder.embed(
    rows, entity_type="gene", id_type="symbol", obs_column="symbol",
    model="genept", output="anndata", key="X_rows",
)

print("obsm keys :", list(rows.obsm.keys()))
print("shape     :", rows.obsm["X_rows"].shape, "→ one row per observation")

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

obsm keys : ['X_rows']
shape     : (6, 3072) → one row per observation


## Feature-aligned embeddings → `.varm`

When you embed the **features** (genes as columns of an expression matrix), the
embedding is feature-aligned and lands in `.varm` — one vector per `var`, so it
stays matched to your gene axis.

In [3]:
feat = ad.AnnData(
    X=np.zeros((1, len(genes)), dtype=np.float32),
    obs=pd.DataFrame(index=["example_cell"]),
    var=pd.DataFrame({"gene_symbol": genes}, index=pd.Index(genes, name="gene_symbol")),
)
feat = embedder.embed(
    feat, entity_type="gene", id_type="symbol", var_column="gene_symbol",
    model="genept", output="anndata", key="X_gene_feature",
)

print("varm keys :", list(feat.varm.keys()))
print("shape     :", feat.varm["X_gene_feature"].shape, "→ one row per gene (var)")

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

varm keys : ['X_gene_feature']
shape     : (6, 3072) → one row per gene (var)


## Payload + provenance → `.uns`

A 384-column matrix does not say what it is. The same three genes, the same
weights, and a different pooling strategy give you a different matrix with no
visible marker of the difference — and if you passed an identifier that embpy
renamed on the way in, the rows are not even keyed by what you typed.

`.uns` is where embpy records the answer: which model, which pooling, which
layer, what you asked for, and what actually came back. The next few cells make
each of those visible in turn.

In [4]:
from embpy.embedder_registry.flat import MODEL_REGISTRY

# minilm_l6_v2 rather than genept for this section: genept is a lookup table,
# so there is nothing to pool. A real encoder emits one vector per token, which
# is what makes the pooling choice meaningful.
wrapper_cls, weights = MODEL_REGISTRY["minilm_l6_v2"]
print("weights  :", weights)
print("modality :", wrapper_cls.model_type)
print("pooling  :", wrapper_cls.available_pooling_strategies)

weights  : sentence-transformers/all-MiniLM-L6-v2
modality : text
pooling  : ['mean', 'max', 'cls', 'last_token', 'none']


### The identifier you type is not always the identifier you get

Two of the four symbols below are stale HGNC names. embpy resolves them to the
approved symbol before anything is embedded, so the row you get back is keyed by
something you never typed — and one symbol does not resolve at all.

In [5]:
requested = ["TP53", "AARS", "EPRS", "NOTAGENE9"]

for sym in requested:
    approved = embedder.gene_resolver.resolve_symbol(sym, organism="human")
    print(f"  you type {sym:10s} → HGNC approved {approved}")

  you type TP53       → HGNC approved TP53
  you type AARS       → HGNC approved AARS1
  you type EPRS       → HGNC approved EPRS1
  you type NOTAGENE9  → HGNC approved None


The rename is not cosmetic bookkeeping — it changes what the model reads. For a
text encoder, embpy embeds the gene's *description*, and the description is
fetched for the approved symbol:

In [6]:
print(embedder.gene_resolver.get_gene_description("AARS", "symbol", "human")[:170])

Gene AARS (human). AARS1: alanyl-tRNA synthetase 1. The human alanyl-tRNA synthetase (AARS) belongs to a family of tRNA synthases, of the class II enzymes. Class II tRNA 


### Embed with average pooling

`pooling_strategy="mean"` averages the per-token vectors into one vector per
gene. Note that no `key=` is passed here: left to itself, embpy names the matrix
after what produced it.

In [7]:
emb = embedder.embed(
    requested, entity_type="gene", id_type="symbol",
    model="minilm_l6_v2", pooling_strategy="mean", output="anndata",
)

print("varm key :", list(emb.varm.keys()))
print()
print(emb.var)

varm key : ['X_emb__gene__minilm_l6_v2__pool_mean']

                gene_symbol
ensembl_gene_id            
ENSG00000141510        TP53
ENSG00000090861        AARS
ENSG00000136628        EPRS


Three things to read off that output.

* **The index is the canonical id**, `ensembl_gene_id` — not the symbol. The
  `gene_symbol` column preserves what you typed, so `AARS` sits next to
  `ENSG00000090861`, which is AARS1. Without that column the rename would be
  unrecoverable.
* **Four in, three out.** `NOTAGENE9` is simply absent. A silently shorter
  matrix is the failure mode worth guarding against, which is why the count is
  recorded rather than left for you to notice.
* **The key encodes the recipe** — `pool_mean` is in the name. Convenient, but
  a key is a nickname, not the record; the next cells show where it stops being
  enough.

### Same genes, same weights, different pooling

Now pool the *same* model's tokens with `cls` instead, into the same object.

In [8]:
emb = embedder.embed(
    requested, entity_type="gene", id_type="symbol",
    model="minilm_l6_v2", pooling_strategy="cls", target=emb, output="anndata",
)

mean_k = "X_emb__gene__minilm_l6_v2__pool_mean"
cls_k = "X_emb__gene__minilm_l6_v2__pool_cls"
print("varm keys:", list(emb.varm.keys()))

a, b = emb.varm[mean_k][0], emb.varm[cls_k][0]
cosine = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print(f"cosine(mean, cls) on TP53: {cosine:.3f}")

varm keys: ['X_emb__gene__minilm_l6_v2__pool_mean', 'X_emb__gene__minilm_l6_v2__pool_cls']
cosine(mean, cls) on TP53: 0.369


Two 384-dim matrices, the same three genes, the same checkpoint, 0.37 apart.
Nothing in either matrix distinguishes them. Hand someone the arrays alone and
the difference is unrecoverable.

### Read the record back

Every matrix embpy writes gets a block in `.uns["embeddings"]`, keyed the same
way as the matrix itself.

In [9]:
for key, block in emb.uns["embeddings"].items():
    p = block["provenance"]
    x = p["extra"]
    print(key)
    print("   model / pooling / layer :", p["model"], "/", p["pooling"], "/", p["layer"])
    print("   rows are                :", block["id_scheme"],
          f'({block["n_entities"]} x {block["n_dims"]})')
    print("   you passed              :", x["input_id_type"], "| organism:", x["organism"])
    print("   kept / requested        :", x["n_successfully_embedded_entities"],
          "/", x["n_requested_inputs"])
    print("   dropped                 :", x["n_dropped_or_unresolved_entities"],
          "(no data:", x["n_embedding_failures"],
          ", unresolvable id:", x["n_unresolved_identifiers"], ")")
    print("   stamped                 :", p["embpy_version"], "|", p["timestamp"])

X_emb__gene__minilm_l6_v2__pool_mean
   model / pooling / layer : minilm_l6_v2 / mean / None
   rows are                : ensembl_gene_id (3 x 384)
   you passed              : symbol | organism: human
   kept / requested        : 3 / 4
   dropped                 : 1 (no data: 1 , unresolvable id: 0 )
   stamped                 : 0.1.dev268+g5badd093f.d20260812 | 2026-08-20T19:10:23+00:00
X_emb__gene__minilm_l6_v2__pool_cls
   model / pooling / layer : minilm_l6_v2 / cls / None
   rows are                : ensembl_gene_id (3 x 384)
   you passed              : symbol | organism: human
   kept / requested        : 3 / 4
   dropped                 : 1 (no data: 1 , unresolvable id: 0 )
   stamped                 : 0.1.dev268+g5badd093f.d20260812 | 2026-08-20T19:10:29+00:00


That is the difference between having an embedding and being able to defend one.
`pooling` separates the two matrices above, `n_requested_inputs` versus
`n_successfully_embedded_entities` accounts for the dropped gene, `id_scheme`
says the rows are Ensembl ids rather than the symbols you passed, and the
version and timestamp say which build made them.

> **The key is a nickname; provenance is the record.** `_default_key` sanitises
> anything that is not alphanumeric, so a negative layer index loses its sign:
> `layer=-2` produces the key `..._pool_mean__layer__2` while provenance still
> reports `layer: -2`. Read the recipe off `.uns`, never off the key.

## Save once, reload anywhere

Because everything lives in the AnnData, a single `write_h5ad` persists your
embeddings *and* their provenance. The check worth making is not that the matrix
came back — a shape is easy to preserve — but that you can still tell the two
matrices apart afterwards. An embedding you cannot attribute is not much better
than one you never computed.

In [10]:
from pathlib import Path

out = Path("outputs"); out.mkdir(exist_ok=True)
emb.write_h5ad(out / "gene_embeddings.h5ad")

reloaded = ad.read_h5ad(out / "gene_embeddings.h5ad")

print("matrices     :", list(reloaded.varm.keys()))
print("canonical ids:", list(reloaded.var_names))
print("as typed     :", list(reloaded.var["gene_symbol"]))

# The point of the round trip: the recipe survives it too, so months later you
# can still say which of these two matrices was mean-pooled.
for key, block in reloaded.uns["embeddings"].items():
    p = block["provenance"]
    print(f"{key:44s} pooling={p['pooling']:5s} rows={block['id_scheme']}")

matrices     : ['X_emb__gene__minilm_l6_v2__pool_cls', 'X_emb__gene__minilm_l6_v2__pool_mean']
canonical ids: ['ENSG00000141510', 'ENSG00000090861', 'ENSG00000136628']
as typed     : ['TP53', 'AARS', 'EPRS']
X_emb__gene__minilm_l6_v2__pool_cls          pooling=cls   rows=ensembl_gene_id
X_emb__gene__minilm_l6_v2__pool_mean         pooling=mean  rows=ensembl_gene_id


## Summary

- Row-aligned → `.obsm`, feature-aligned → `.varm`, payload/provenance → `.uns`.
- `.X` is always your data, never an embedding.
- One `write_h5ad` saves the vectors and how they were made.

**Next:** [Comparing embeddings](03_compare_embeddings.ipynb).